### Statistic analysis

In [ ]:
from pathlib import Path

# code = r'''
# ============================================================
# COMPLETE STATISTICAL ANALYSIS
# Encoders: GIN, GINE, GCNRes
# Datasets: Cora, ACM-DBLP, Douban
# Includes:
#   1) Descriptive statistics
#   2) Friedman tests
#   3) Wilcoxon signed-rank + Holm correction
#   4) Average ranks
#   5) Detailed tables by Dataset × Dimension × Layer
#      with mean ± sample standard deviation over 10 seeds
#   6) CSV and LaTeX exports
# ============================================================

# average_ranks.csv: identifica el encoder con mejor ranking promedio. Para Hits@1, Hits@10 y MRR, un ranking menor es mejor; para Runtime, también es menor el tiempo más conveniente.

# friedman_results.csv: verifica si la diferencia global entre encoders es estadísticamente significativa, usando \(p < 0.05\) como referencia.

# posthoc_wilcoxon_holm.csv: si Friedman detecta diferencias significativas, comprueba qué pares de encoders difieren significativamente.

# Las tablas de rendimiento por dataset: comprueba si la conclusión global también se mantiene en Cora, ACM-DBLP y Douban.

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

# ============================================================
# 1. PARAMETERS
# ============================================================

FILE = "experiments/output/Final_experiments-Summary.csv"

OUTPUT_DIR = Path("analysis_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENCODERS = ["GIN", "GINE", "GCNRes"]
DATASETS = ["Cora", "ACM-DBLP", "Douban"]

METRICS = ["Hits@1", "Hits@10", "MRR", "Runtime"]
PERFORMANCE_METRICS = ["Hits@1", "Hits@10", "MRR"]

# Higher is better for retrieval metrics; lower is better for runtime.
HIGHER_IS_BETTER = {
    "Hits@1": True,
    "Hits@10": True,
    "MRR": True,
    "Runtime": False,
}

# ============================================================
# 2. LOAD AND VALIDATE DATA
# ============================================================

df = pd.read_csv(FILE, decimal=",")

required_columns = [
    "Dataset", "Encoder", "Dimension", "Layer", "Seed",
    "Accuracy", "Hits@1", "Hits@10", "Hits@30", "Hits@50",
    "MRR", "Runtime"
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in input CSV: {missing}")

numeric_columns = [
    "Dimension", "Layer", "Seed", "Accuracy",
    "Hits@1", "Hits@10", "Hits@30", "Hits@50", "MRR", "Runtime"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset shape:", df.shape)
print("Rows per Dataset × Encoder:")
print(df.groupby(["Dataset", "Encoder"]).size())

# Check number of seeds per configuration
seed_counts = (
    df.groupby(["Dataset", "Dimension", "Layer", "Encoder"])["Seed"]
      .nunique()
)
if not (seed_counts == 10).all():
    print("\nWARNING: Some configurations do not contain exactly 10 unique seeds.")
    print(seed_counts[seed_counts != 10])

Dataset shape: (2880, 12)
Rows per Dataset × Encoder:
Dataset   Encoder
ACM-DBLP  GCNRes     320
          GIN        320
          GINE       320
Cora      GCNRes     320
          GIN        320
          GINE       320
Douban    GCNRes     320
          GIN        320
          GINE       320
dtype: int64


In [8]:
# ============================================================
# 3. DESCRIPTIVE STATISTICS ON RAW RUNS
# ============================================================

descriptive = (
    df.groupby(["Dataset", "Encoder"])[METRICS]
      .agg(["mean", "std"])
)
descriptive.to_csv(OUTPUT_DIR / "descriptive_statistics_raw_runs.csv")
descriptive

Hits@1             Hits@10                 MRR            \
                      mean       std      mean       std      mean       std   
Dataset  Encoder                                                               
ACM-DBLP GCNRes   0.566495  0.058875  0.859234  0.033415  0.665840  0.050509   
         GIN      0.559303  0.054282  0.840434  0.031898  0.654064  0.046693   
         GINE     0.569943  0.048184  0.854399  0.032557  0.665772  0.043221   
Cora     GCNRes   0.860215  0.138868  0.961407  0.043069  0.898332  0.105642   
         GIN      0.881429  0.155293  0.971677  0.052965  0.915964  0.119779   
         GINE     0.884864  0.153316  0.971567  0.051651  0.917746  0.118260   
Douban   GCNRes   0.237121  0.019222  0.539732  0.028176  0.337358  0.021812   
         GIN      0.314884  0.043482  0.567221  0.052875  0.401203  0.046300   
         GINE     0.223933  0.035463  0.452477  0.058577  0.300858  0.042642   

                     Runtime             
                        mean        std  
Dataset  Encoder                         
ACM-DBLP GCNRes    39.163939  21.199186  
         GIN       69.554390  35.684374  
         GINE     131.347128  83.837210  
Cora     GCNRes    10.490187   6.284377  
         GIN       24.420633  15.402681  
         GINE      36.313623  24.167998  
Douban   GCNRes     5.910367   3.381788  
         GIN       12.852045   6.737236  
         GINE      21.820819  15.036496

In [9]:
# ============================================================
# 4. CONFIGURATION-LEVEL MEANS FOR INFERENTIAL TESTS
# ============================================================
# First average the 10 seeds within each:
# Dataset × Dimension × Layer × Encoder.
# These configuration-level means are used for paired tests.

config_cols = ["Dataset", "Dimension", "Layer", "Encoder"]

config_df = (
    df.groupby(config_cols)[METRICS]
      .mean()
      .reset_index()
)
config_df

,Dataset,Dimension,Layer,Encoder,Hits@1,Hits@10,MRR,Runtime
0,ACM-DBLP,8,2,GCNRes,0.49071,0.81924,0.60216,15.64780
1,ACM-DBLP,8,2,GIN,0.47383,0.79988,0.58347,22.25222
2,ACM-DBLP,8,2,GINE,0.48113,0.80032,0.58828,34.20010
3,ACM-DBLP,8,4,GCNRes,0.42535,0.77830,0.54450,17.57712
4,ACM-DBLP,8,4,GIN,0.44128,0.77525,0.55291,43.23904
...,...,...,...,...,...,...,...,...
283,Douban,1024,8,GIN,0.35666,0.61786,0.44555,20.41268
284,Douban,1024,8,GINE,0.24929,0.47984,0.32611,28.18023
285,Douban,1024,12,GCNRes,0.24445,0.54633,0.34675,15.01291
286,Douban,1024,12,GIN,0.34759,0.60492,0.43493,24.25426


In [10]:
# ============================================================
# 5. FRIEDMAN TEST
# ============================================================

def make_pivot(data, metric, dataset=None):
    if dataset is not None:
        data = data[data["Dataset"] == dataset]

    pivot = data.pivot(
        index=["Dataset", "Dimension", "Layer"],
        columns="Encoder",
        values=metric
    )

    # Ensure the encoder columns are in the same order.
    pivot = pivot.reindex(columns=["GCNRes", "GIN", "GINE"])
    return pivot.dropna()


def run_friedman(data, metric, dataset=None):
    pivot = make_pivot(data, metric, dataset)

    statistic, p_value = friedmanchisquare(
        pivot["GCNRes"].to_numpy(),
        pivot["GIN"].to_numpy(),
        pivot["GINE"].to_numpy()
    )
    return statistic, p_value, pivot


# ============================================================
# 6. WILCOXON POST-HOC WITH HOLM CORRECTION
# ============================================================

def posthoc_wilcoxon(pivot, metric):
    pairs = [
        ("GIN", "GINE"),
        ("GIN", "GCNRes"),
        ("GINE", "GCNRes")
    ]

    rows = []
    raw_pvalues = []

    for encoder_a, encoder_b in pairs:
        x = pivot[encoder_a].to_numpy()
        y = pivot[encoder_b].to_numpy()

        # Handle the unlikely case where all paired differences are zero.
        if np.allclose(x - y, 0):
            statistic, p_value = 0.0, 1.0
        else:
            statistic, p_value = wilcoxon(
                x, y, alternative="two-sided", zero_method="wilcox"
            )

        raw_pvalues.append(p_value)
        rows.append({
            "Metric": metric,
            "Comparison": f"{encoder_a} vs {encoder_b}",
            "Statistic": statistic,
            "p_raw": p_value
        })

    reject, p_adjusted, _, _ = multipletests(
        raw_pvalues, method="holm"
    )

    for i in range(len(rows)):
        rows[i]["p_adjusted_Holm"] = p_adjusted[i]
        rows[i]["Significant_0.05"] = bool(reject[i])

    return pd.DataFrame(rows)


# ============================================================
# 7. AVERAGE RANKS
# ============================================================

def average_ranks(pivot, higher_is_better=True):
    ranks = pivot.copy()

    ranks = ranks.rank(
        axis=1,
        ascending=not higher_is_better,
        method="average"
    )

    return ranks.mean().sort_values()


# ============================================================
# 8. FRIEDMAN ANALYSIS BY DATASET AND GLOBALLY
# ============================================================

friedman_rows = []
posthoc_rows = []
rank_rows = []

for metric in METRICS:
    for dataset in DATASETS:
        statistic, p_value, pivot = run_friedman(
            config_df, metric, dataset
        )

        friedman_rows.append({
            "Scope": dataset,
            "Metric": metric,
            "N_blocks": len(pivot),
            "Friedman_chi2": statistic,
            "p_value": p_value,
            "Significant_0.05": bool(p_value < 0.05)
        })

        ranks = average_ranks(
            pivot, HIGHER_IS_BETTER[metric]
        )
        rank_rows.append({
            "Scope": dataset,
            "Metric": metric,
            "GCNRes_rank": ranks.get("GCNRes", np.nan),
            "GIN_rank": ranks.get("GIN", np.nan),
            "GINE_rank": ranks.get("GINE", np.nan),
            "Best_average_rank_encoder": ranks.index[0]
        })

        # Standard practice: run post-hoc only if the omnibus
        # Friedman test is significant.
        if p_value < 0.05:
            posthoc = posthoc_wilcoxon(pivot, metric)
            posthoc.insert(0, "Dataset", dataset)
            posthoc.insert(1, "Scope", dataset)
            posthoc["Friedman_p_value"] = p_value
            posthoc_rows.append(posthoc)

    # Global comparison: all Dataset × Dimension × Layer blocks
    statistic, p_value, pivot = run_friedman(config_df, metric)

    friedman_rows.append({
        "Scope": "Global",
        "Metric": metric,
        "N_blocks": len(pivot),
        "Friedman_chi2": statistic,
        "p_value": p_value,
        "Significant_0.05": bool(p_value < 0.05)
    })

    ranks = average_ranks(
        pivot, HIGHER_IS_BETTER[metric]
    )
    rank_rows.append({
        "Scope": "Global",
        "Metric": metric,
        "GCNRes_rank": ranks.get("GCNRes", np.nan),
        "GIN_rank": ranks.get("GIN", np.nan),
        "GINE_rank": ranks.get("GINE", np.nan),
        "Best_average_rank_encoder": ranks.index[0]
    })

    if p_value < 0.05:
        posthoc = posthoc_wilcoxon(pivot, metric)
        posthoc.insert(0, "Dataset", "All datasets")
        posthoc.insert(1, "Scope", "Global")
        posthoc["Friedman_p_value"] = p_value
        posthoc_rows.append(posthoc)

friedman_table = pd.DataFrame(friedman_rows)
friedman_table.to_csv(
    OUTPUT_DIR / "friedman_results.csv", index=False
)

rank_table = pd.DataFrame(rank_rows)
rank_table.to_csv(
    OUTPUT_DIR / "average_ranks.csv", index=False
)

if posthoc_rows:
    posthoc_table = pd.concat(posthoc_rows, ignore_index=True)
    posthoc_table.to_csv(
        OUTPUT_DIR / "posthoc_wilcoxon_holm.csv", index=False
    )

# ============================================================
# 9. DETAILED TABLES:
#    EACH DATASET × DIMENSION × LAYER
#    MEAN ± SAMPLE SD OVER THE 10 SEEDS
# ============================================================
# For each configuration, compute the mean and sample standard
# deviation (ddof=1) across its 10 seeds. Each output cell is:
# mean ± standard deviation for that exact configuration.
#
# The table has 32 rows per dataset and 11 columns:
# Dimension, Layer, and 3 encoders for each of 3 metrics.
# Runtime is also exported in a separate table for readability.

detailed_metrics = PERFORMANCE_METRICS + ["Runtime"]

config_stats = (
    df.groupby(config_cols)[detailed_metrics]
      .agg(["mean", "std"])
      .reset_index()
)

# Flatten the MultiIndex column names, e.g. ("Hits@1", "mean")
flat_cols = []
for col in config_stats.columns:
    if isinstance(col, tuple):
        flat_cols.append("_".join(str(part) for part in col if part != ""))
    else:
        flat_cols.append(str(col))
config_stats.columns = flat_cols

config_stats.to_csv(
    OUTPUT_DIR / "configuration_means_and_standard_deviations.csv",
    index=False
)


def format_mean_sd(mean_value, std_value, decimals=4):
    if pd.isna(mean_value) or pd.isna(std_value):
        return "NA"
    # Use decimal comma in the displayed table.
    return (
        f"{mean_value:.{decimals}f} ± {std_value:.{decimals}f}"
        .replace(".", ",")
    )


def make_dataset_table(dataset, metrics_to_include):
    subset = config_stats[config_stats["Dataset"] == dataset].copy()
    subset = subset.sort_values(["Dimension", "Layer"])

    rows = []
    for dimension in sorted(subset["Dimension"].unique()):
        for layer in sorted(subset.loc[subset["Dimension"] == dimension, "Layer"].unique()):
            row = {
                "Dimension": int(dimension),
                "Layer": int(layer)
            }

            for metric in metrics_to_include:
                for encoder in ENCODERS:
                    selected = subset[
                        (subset["Dimension"] == dimension)
                        & (subset["Layer"] == layer)
                        & (subset["Encoder"] == encoder)
                    ]

                    if selected.empty:
                        value = "NA"
                    else:
                        record = selected.iloc[0]
                        value = format_mean_sd(
                            record[f"{metric}_mean"],
                            record[f"{metric}_std"]
                        )

                    row[f"{metric} - {encoder}"] = value

            rows.append(row)

    return pd.DataFrame(rows)


def write_latex_table(table, dataset, metrics_to_include, filename):
    # Multi-column grouped headers for paper-ready LaTeX.
    n_metric_cols = len(metrics_to_include) * len(ENCODERS)
    lines = [
        r"\begin{table*}[ht]",
        r"\centering",
        rf"\caption{{Results on {dataset}: mean $\pm$ sample standard deviation over 10 seeds.}}",
        rf"\label{{tab:results-{dataset.lower().replace('-', '')}}}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{rr" + "c" * n_metric_cols + r"}",
        r"\toprule",
        r"\multirow{2}{*}{\textbf{Dimension}} & "
        r"\multirow{2}{*}{\textbf{Layer}} & "
        + " & ".join(
            rf"\multicolumn{{3}}{{c}}{{\textbf{{{metric}}}}}"
            for metric in metrics_to_include
        )
        + r" \\",
    ]

    cmidrules = []
    start = 3
    for _ in metrics_to_include:
        cmidrules.append(rf"\cmidrule(lr){{{start}-{start+2}}}")
        start += 3
    lines.append(" ".join(cmidrules))

    encoder_headers = []
    for _metric in metrics_to_include:
        encoder_headers.extend([rf"\textbf{{{encoder}}}" for encoder in ENCODERS])
    lines.append(" & & " + " & ".join(encoder_headers) + r" \\")
    lines.append(r"\midrule")

    for _, row in table.iterrows():
        cells = [str(int(row["Dimension"])), str(int(row["Layer"]))]
        for metric in metrics_to_include:
            for encoder in ENCODERS:
                val = row[f"{metric} - {encoder}"]
                val = val.replace("±", r"$\pm$")
                cells.append(val)
        lines.append(" & ".join(cells) + r" \\")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}%",
        r"}",
        r"\end{table*}"
    ])

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


for dataset in DATASETS:
    # Table A: retrieval metrics, matching the requested layout
    perf_table = make_dataset_table(dataset, PERFORMANCE_METRICS)
    perf_csv = OUTPUT_DIR / f"performance_{dataset.replace('-', '_')}_mean_sd_10_seeds.csv"
    perf_table.to_csv(perf_csv, index=False)

    perf_tex = OUTPUT_DIR / f"performance_{dataset.replace('-', '_')}_mean_sd_10_seeds.tex"
    write_latex_table(perf_table, dataset, PERFORMANCE_METRICS, perf_tex)

    # Table B: runtime per configuration
    runtime_table = make_dataset_table(dataset, ["Runtime"])
    runtime_csv = OUTPUT_DIR / f"runtime_{dataset.replace('-', '_')}_mean_sd_10_seeds.csv"
    runtime_table.to_csv(runtime_csv, index=False)

    runtime_tex = OUTPUT_DIR / f"runtime_{dataset.replace('-', '_')}_mean_sd_10_seeds.tex"
    write_latex_table(runtime_table, dataset, ["Runtime"], runtime_tex)

# One LaTeX file containing the three retrieval tables
combined_tex = OUTPUT_DIR / "all_datasets_performance_tables.tex"
with open(combined_tex, "w", encoding="utf-8") as outfile:
    for dataset in DATASETS:
        part = OUTPUT_DIR / f"performance_{dataset.replace('-', '_')}_mean_sd_10_seeds.tex"
        outfile.write(part.read_text(encoding="utf-8"))
        outfile.write("\n\n")

# ============================================================
# 10. RAW MEANS BY DATASET × ENCODER
# ============================================================

raw_means = (
    df.groupby(["Dataset", "Encoder"])[METRICS]
      .mean()
      .reset_index()
)
raw_means.to_csv(
    OUTPUT_DIR / "raw_means_by_dataset_encoder.csv", index=False
)

# ============================================================
# 11. REPORT OUTPUTS
# ============================================================

print("\nAnalysis finished.")
print(f"All output files are saved in: {OUTPUT_DIR.resolve()}")
print("\nMain output files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" -", path.name)

print("\nFriedman results:")
print(friedman_table.to_string(index=False))

print("\nAverage ranks:")
print(rank_table.to_string(index=False))

# path = Path("/mnt/data/complete_encoder_statistical_analysis.py")
#path.write_text(code, encoding="utf-8")
print(f"Created Python script: {path}")


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  res = hypotest_fun_out(*samples, **kwds)



Analysis finished.
All output files are saved in: /home/ealvarez/Project/T-GAE_PyG/analysis_results

Main output files:
 - all_datasets_performance_tables.tex
 - average_ranks.csv
 - configuration_means_and_standard_deviations.csv
 - descriptive_statistics_raw_runs.csv
 - friedman_results.csv
 - performance_ACM_DBLP_mean_sd_10_seeds.csv
 - performance_ACM_DBLP_mean_sd_10_seeds.tex
 - performance_Cora_mean_sd_10_seeds.csv
 - performance_Cora_mean_sd_10_seeds.tex
 - performance_Douban_mean_sd_10_seeds.csv
 - performance_Douban_mean_sd_10_seeds.tex
 - posthoc_wilcoxon_holm.csv
 - raw_means_by_dataset_encoder.csv
 - runtime_ACM_DBLP_mean_sd_10_seeds.csv
 - runtime_ACM_DBLP_mean_sd_10_seeds.tex
 - runtime_Cora_mean_sd_10_seeds.csv
 - runtime_Cora_mean_sd_10_seeds.tex
 - runtime_Douban_mean_sd_10_seeds.csv
 - runtime_Douban_mean_sd_10_seeds.tex

Friedman results:
   Scope  Metric  N_blocks  Friedman_chi2      p_value  Significant_0.05
    Cora  Hits@1        32      27.228346 1.223037e-06  

---